In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,141,35.416449
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,141,36.912919
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,141,37.380566
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,141,35.291744
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,141,30.771158
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,0,129,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,0,129,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,0,129,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,0,129,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.485855e+06
              2                  2.580952e+06
              3                  2.255781e+06
              4                  2.010391e+06
              5                  1.527952e+06
intervention  1                  2.485861e+06
              2                  2.580964e+06
              3                  2.255795e+06
              4                  2.010404e+06
              5                  1.527966e+06
zero          1                  2.485855e+06
              2                  2.580952e+06
              3                  2.255781e+06
              4                  2.010391e+06
              5                  1.527952e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.312310e+06
              2                  1.328116e+06
              3                  1.126331e+06
              4                  1.020841e+06
              5                  6.935400e+05
intervention  1                  1.275864e+06
              2                  1.276266e+06
              3                  1.061637e+06
              4                  9.642470e+05
              5                  6.325914e+05
zero          1                  1.312310e+06
              2                  1.328116e+06
              3                  1.126331e+06
              4                  1.020841e+06
              5                  6.935400e+05
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.527911
              2                  0.514584
              3                  0.499308
              4                  0.507782
              5                  0.453902
intervention  1                  0.513248
              2                  0.494492
              3                  0.470627
              4                  0.479628
              5                  0.414009
zero          1                  0.527911
              2                  0.514584
              3                  0.499308
              4                  0.507782
              5                  0.453902
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,18793.149621
1,Female,0.0,0.019178,not_pregnant,2,17830.620200
2,Female,0.0,0.019178,not_pregnant,3,16129.662368
3,Female,0.0,0.019178,not_pregnant,4,14114.028154
4,Female,0.0,0.019178,not_pregnant,5,12208.445301
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,4849.649774
281,Male,95.0,125.000000,not_pregnant,2,4371.181536
282,Male,95.0,125.000000,not_pregnant,3,4569.918352
283,Male,95.0,125.000000,not_pregnant,4,4550.906874


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    2.306716e+06
2    2.394711e+06
3    2.088829e+06
4    1.862288e+06
5    1.411894e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.217740e+06
              2                  1.232280e+06
              3                  1.042970e+06
              4                  9.456368e+05
              5                  6.408611e+05
intervention  1                  1.183918e+06
              2                  1.184165e+06
              3                  9.830583e+05
              4                  8.932060e+05
              5                  5.845367e+05
zero          1                  1.217740e+06
              2                  1.232280e+06
              3                  1.042970e+06
              4                  9.456368e+05
              5                  6.408611e+05
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,141,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,141,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,141,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,141,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,0,129,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,0,129,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,0,129,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,0,129,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.561085e+06
              2                  2.216806e+06
              3                  2.114086e+06
              4                  1.925897e+06
              5                  1.405792e+06
intervention  1                  2.539436e+06
              2                  2.186236e+06
              3                  2.077616e+06
              4                  1.890654e+06
              5                  1.374996e+06
zero          1                  2.561085e+06
              2                  2.216806e+06
              3                  2.114086e+06
              4                  1.925897e+06
              5                  1.405792e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,146.630818
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,172.698519
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,164.552362
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,148.260049
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,109.158498
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,166.181593
23996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,149.889280
23997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,164.552362
23998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,105.900035


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  172903.801774
              2                  179047.633036
              3                  158763.703254
              4                  140314.287924
              5                  106730.942976
intervention  1                  172864.700223
              2                  178993.868403
              3                  158677.353994
              4                  140255.635597
              5                  106638.076791
zero          1                  172903.801774
              2                  179047.633036
              3                  158763.703254
              4                  140314.287924
              5                  106730.942976
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,15253.787430,zero
1,Female,0.0,0.019178,2,14177.057673,zero
2,Female,0.0,0.019178,3,12677.857494,zero
3,Female,0.0,0.019178,4,10921.063707,zero
4,Female,0.0,0.019178,5,8999.215869,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,3594.777472,intervention
746,Male,95.0,125.000000,2,3266.016080,intervention
747,Male,95.0,125.000000,3,3449.778426,intervention
748,Male,95.0,125.000000,4,3437.070165,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.395973e+07
              2                  2.348450e+07
              3                  2.171342e+07
              4                  2.046778e+07
              5                  1.920935e+07
intervention  1                  2.336751e+07
              2                  2.267088e+07
              3                  2.066526e+07
              4                  1.925104e+07
              5                  1.790954e+07
zero          1                  2.395973e+07
              2                  2.348450e+07
              3                  2.171342e+07
              4                  2.046778e+07
              5                  1.920935e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.517747e+07
              2                  2.471678e+07
              3                  2.275639e+07
              4                  2.141342e+07
              5                  1.985021e+07
intervention  1                  2.455142e+07
              2                  2.385505e+07
              3                  2.164832e+07
              4                  2.014424e+07
              5                  1.849407e+07
zero          1                  2.517747e+07
              2                  2.471678e+07
              3                  2.275639e+07
              4                  2.141342e+07
              5                  1.985021e+07
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  12734.955657
              2                  13022.284791
              3                  11788.751094
              4                  10449.533117
              5                   9178.019131
baseline      1                  12734.955657
              2                  13022.284791
              3                  11788.751094
              4                  10449.533117
              5                   9178.019131
intervention  1                  11753.394822
              2                  11219.337991
              3                   9618.862519
              4                   8011.556809
              5                   6848.115308
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)